# Task 7 — Explain Every Flag: Reason Codes and Money at Risk

**Goal:** Turn Task 6 payroll anomaly predictions into clear, auditable explanations in the official `sample_anomaly_results.csv` output format.

This notebook:
1. Reads the sample output to preserve the exact 12-column schema.
2. Loads Task 6 payroll predictions and the payroll feature table.
3. Maps transparent detector/rule signals to clerk-friendly reason codes and sentences.
4. Estimates rupees at risk from the payroll amount fields available in the feature table.
5. Ranks flagged rows by money at risk, while retaining the official output columns.
6. Validates the output schema and produces a 20-row review sample.

**Important:** This notebook does not read or use any hidden validation-label/unlock file. Money at risk is an operational estimate, not proof of actual loss. If a row has no reliable amount field or no explainable signal, the notebook marks the reason as a reviewable statistical outlier and leaves money at risk blank rather than inventing a rupee amount.


In [160]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Resolve project root whether this notebook is in notebooks/ or the repository root.
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
ROOT = next(
    (p for p in CANDIDATE_ROOTS if (p / "reports").exists() and (p / "data").exists()),
    next((p for p in CANDIDATE_ROOTS if (p / "reports").exists()), Path.cwd())
)
print("Project root:", ROOT)


Project root: /workspace


## 1. Load the official sample output format

Keep the column names and order exactly as supplied. The sample is a schema reference; it is not the source of new predictions.


In [161]:
SAMPLE_CANDIDATES = [
    ROOT / "data" / "sample_anomaly_results.csv",
    ROOT / "sample_anomaly_results.csv",
    ROOT / "data" / "reference" / "sample_anomaly_results.csv",
]
sample_path = next((p for p in SAMPLE_CANDIDATES if p.exists()), None)
if sample_path is None:
    # Optional recursive fallback for projects that store the sample in another data subfolder.
    matches = list(ROOT.glob("**/sample_anomaly_results.csv"))
    sample_path = matches[0] if matches else None
if sample_path is None:
    raise FileNotFoundError(
        "Could not find sample_anomaly_results.csv. Put it in the project root or data/ folder."
    )

sample = pd.read_csv(sample_path)
OFFICIAL_COLUMNS = sample.columns.tolist()
print("Sample file:", sample_path)
print("Sample shape:", sample.shape)
print("Official output columns:", OFFICIAL_COLUMNS)
display(sample.head())


Sample file: /workspace/data/sample_anomaly_results.csv
Sample shape: (405, 12)
Official output columns: ['result_id', 'source_dataset', 'record_id', 'employee_id', 'period', 'anomaly_score', 'flagged', 'reason_code', 'reason_text', 'money_at_risk', 'model_version', 'scored_at']


,result_id,source_dataset,record_id,employee_id,period,anomaly_score,flagged,reason_code,reason_text,money_at_risk,model_version,scored_at
0,AN-0001,attendance_monthly_features,AF-03170,EMP-0356,2025-07-01,1.000,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00
1,AN-0002,attendance_monthly_features,AF-05918,EMP-0585,2025-09-01,0.998,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00
2,AN-0003,attendance_monthly_features,AF-06802,EMP-0646,2026-06-01,0.998,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00
3,AN-0004,attendance_monthly_features,AF-02819,EMP-0318,2026-06-01,0.998,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00
4,AN-0005,attendance_monthly_features,AF-03112,EMP-0347,2026-09-01,0.992,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00


## 2. Load Task 6 predictions and payroll features

Expected Task 6 prediction file: `reports/task6_payroll_anomaly_predictions.csv`.

The feature table is searched in common project locations. If your repository uses a different filename/path, edit `FEATURE_CANDIDATES` below. The code uses record/employee/period keys to join predictions to payroll amounts and explanatory signals.


In [162]:
PREDICTION_CANDIDATES = [
    ROOT / "reports" / "task6_payroll_anomaly_predictions.csv",
    ROOT / "reports" / "task6_payroll_anomaly_predictions_frozen.csv",
]
prediction_path = next((p for p in PREDICTION_CANDIDATES if p.exists()), None)
if prediction_path is None:
    raise FileNotFoundError(
        "Task 6 prediction CSV not found under reports/. Keep the Task 6 output at "
        "reports/task6_payroll_anomaly_predictions.csv."
    )
pred = pd.read_csv(prediction_path)
print("Prediction file:", prediction_path)
print("Prediction shape:", pred.shape)
print("Prediction columns:", pred.columns.tolist())
display(pred.head())


Prediction file: /workspace/reports/task6_payroll_anomaly_predictions.csv
Prediction shape: (7355, 22)
Prediction columns: ['payroll_item_id', 'run_id', 'employee_id', 'month', 'department_id', 'role_id', 'monthly_fixed_gross', 'gross_pay', 'total_deductions', 'net_pay', 'gross_vs_fixed', 'deduction_ratio', 'net_ratio', 'net_reconciliation_gap', 'duplicate_payment_rule', 'negative_net_pay', 'deductions_over_gross', 'reconciliation_rule', 'anomaly_score', 'model_anomaly_flag', 'rules_anomaly_flag', 'combined_anomaly_flag']


,payroll_item_id,run_id,employee_id,month,department_id,role_id,monthly_fixed_gross,gross_pay,total_deductions,net_pay,...,net_ratio,net_reconciliation_gap,duplicate_payment_rule,negative_net_pay,deductions_over_gross,reconciliation_rule,anomaly_score,model_anomaly_flag,rules_anomaly_flag,combined_anomaly_flag
0,PI-000001,PR-2025-04,EMP-0002,2025-04-01,D08,R047,144460.0,179130.40,20625.79,158504.61,...,0.884856,0.000000e+00,0,0,0,0,-0.151812,0,0,0
1,PI-000002,PR-2025-04,EMP-0004,2025-04-01,D03,R017,103938.0,160064.52,31517.82,128546.70,...,0.803093,-1.455192e-11,0,0,0,0,-0.061570,0,0,0
2,PI-000003,PR-2025-04,EMP-0006,2025-04-01,D00,R001,479009.0,479009.00,113100.81,365908.19,...,0.763886,0.000000e+00,0,0,0,0,-0.080985,0,0,0
3,PI-000004,PR-2025-04,EMP-0011,2025-04-01,D07,R039,24041.0,36969.21,1334.69,35634.52,...,0.963897,0.000000e+00,0,0,0,0,-0.144377,0,0,0
4,PI-000005,PR-2025-04,EMP-0012,2025-04-01,D01,R004,53915.0,53915.00,2000.00,51915.00,...,0.962905,0.000000e+00,0,0,0,0,-0.244220,0,0,0


In [163]:
FEATURE_CANDIDATES = [
    ROOT / "data" / "payroll_monthly_features.csv",
    ROOT / "data" / "raw" / "payroll_monthly_features.csv",
    ROOT / "payroll_monthly_features.csv",
    ROOT / "data" / "processed" / "payroll_monthly_features.csv",
]
feature_path = next((p for p in FEATURE_CANDIDATES if p.exists()), None)
if feature_path is None:
    # Search recursively but exclude virtual environments and git internals.
    matches = [
        p for p in ROOT.glob("**/payroll_monthly_features*.csv")
        if ".git" not in p.parts and ".venv" not in p.parts and "venv" not in p.parts
    ]
    feature_path = matches[0] if matches else None
if feature_path is None:
    raise FileNotFoundError(
        "Could not locate payroll_monthly_features*.csv. Place the file in data/ or edit "
        "FEATURE_CANDIDATES to its actual path."
    )
features = pd.read_csv(feature_path)
print("Feature file:", feature_path)
print("Feature shape:", features.shape)
print("Feature columns:", features.columns.tolist())
display(features.head(3))


Feature file: /workspace/data/payroll_monthly_features.csv
Feature shape: (7355, 23)
Feature columns: ['payroll_item_id', 'run_id', 'employee_id', 'month', 'department_id', 'role_id', 'job_level', 'overtime_eligible', 'monthly_fixed_gross', 'payable_days', 'lop_days', 'overtime_hours_paid', 'overtime_pay', 'variable_pay', 'variable_pay_type', 'gross_pay', 'pf_employee', 'esi_employee', 'professional_tax', 'tds', 'other_deductions', 'total_deductions', 'net_pay']


,payroll_item_id,run_id,employee_id,month,department_id,role_id,job_level,overtime_eligible,monthly_fixed_gross,payable_days,...,variable_pay,variable_pay_type,gross_pay,pf_employee,esi_employee,professional_tax,tds,other_deductions,total_deductions,net_pay
0,PI-000001,PR-2025-04,EMP-0002,2025-04-01,D08,R047,4,N,"144,460.00",30.0,...,"34,670.40",Performance Bonus,"179,130.40","1,800.00",0.0,200.0,"18,625.79",0.00,"20,625.79","158,504.61"
1,PI-000002,PR-2025-04,EMP-0004,2025-04-01,D03,R017,3,N,"103,938.00",30.0,...,"56,126.52",Performance Bonus,"160,064.52","1,800.00",0.0,200.0,"29,517.82",0.00,"31,517.82","128,546.70"
2,PI-000003,PR-2025-04,EMP-0006,2025-04-01,D00,R001,7,N,"479,009.00",30.0,...,0.00,NaN,"479,009.00","1,800.00",0.0,200.0,"111,100.81",0.00,"113,100.81","365,908.19"


## 3. Identify the Task 6 flag and join keys

This section accommodates the Task 6 output names used in the project. It fails loudly if it cannot identify a flag column, rather than silently producing an empty result.


In [164]:
# Section 3: Identify anomaly flags and join payroll records
def first_existing(columns, candidates):
    return next((c for c in candidates if c in columns), None)

flag_col = first_existing(
    pred.columns,
    ["combined_anomaly_flag", "anomaly_flag", "flagged", "model_anomaly_flag"],
)
if flag_col is None:
    raise ValueError(
        f"No anomaly flag column found. Available columns: {pred.columns.tolist()}"
    )
if "payroll_item_id" not in pred.columns or "payroll_item_id" not in features.columns:
    raise KeyError("Both Task 6 predictions and payroll features must contain payroll_item_id.")

# Work from clean copies. Preserve the prediction row order and source IDs.
pred_source = pred.copy().reset_index(drop=True)
features_source = features.copy().reset_index(drop=True)
pred_source["payroll_item_id"] = pred_source["payroll_item_id"].astype("string").str.strip()
features_source["payroll_item_id"] = features_source["payroll_item_id"].astype("string").str.strip()

if pred_source["payroll_item_id"].isna().any() or pred_source["payroll_item_id"].eq("").any():
    raise ValueError("Task 6 predictions contain missing payroll_item_id values.")
if features_source["payroll_item_id"].isna().any() or features_source["payroll_item_id"].eq("").any():
    raise ValueError("Payroll features contain missing payroll_item_id values.")
if not pred_source["payroll_item_id"].is_unique:
    raise ValueError("Task 6 predictions contain duplicate payroll_item_id values.")
if not features_source["payroll_item_id"].is_unique:
    raise ValueError("Payroll features contain duplicate payroll_item_id values.")

feature_cols = [
    c for c in features_source.columns
    if c != "payroll_item_id" and c not in pred_source.columns
]
work = pred_source.merge(
    features_source[["payroll_item_id"] + feature_cols],
    on="payroll_item_id",
    how="left",
    validate="one_to_one",
    indicator=True,
    sort=False,
)
matched = int(work["_merge"].eq("both").sum())
unmatched = int(work["_merge"].eq("left_only").sum())
print("Rows after merge:", len(work))
print("Matched rows:", matched)
print("Unmatched rows:", unmatched)
if unmatched:
    missing_ids = work.loc[work["_merge"].eq("left_only"), "payroll_item_id"].head(10).tolist()
    raise ValueError(f"{unmatched} prediction rows did not match payroll features. Sample IDs: {missing_ids}")

# Reassign from the prediction source to guarantee the original identifiers survive the merge.
work["payroll_item_id"] = pred_source["payroll_item_id"].to_numpy()
work = work.drop(columns="_merge")
assert len(work) == len(pred_source)
assert work["payroll_item_id"].notna().all()
assert work["payroll_item_id"].astype(str).str.strip().ne("").all()
print("Valid payroll IDs in work:", int(work["payroll_item_id"].notna().sum()))
print("First payroll IDs:", work["payroll_item_id"].head().tolist())


Rows after merge: 7355
Matched rows: 7355
Unmatched rows: 0
Valid payroll IDs in work: 7355
First payroll IDs: ['PI-000001', 'PI-000002', 'PI-000003', 'PI-000004', 'PI-000005']


## 4. Define transparent reason codes

Reason selection prioritizes direct, interpretable rules. A model-only flag gets `STATISTICAL_OUTLIER` and is described as an unusual payroll pattern requiring review—not as a confirmed error.


In [166]:
# Convert monetary/quantity columns safely without touching identifiers.
# IMPORTANT: payroll_item_id contains the word "payroll"; it must remain a string.
numeric_tokens = (
    "gross", "net", "deduction", "fixed", "salary", "overtime",
    "amount", "allowance", "pf", "tax", "reconciliation_gap",
    "paid", "payment", "expected", "hours"
)
identifier_tokens = ("_id", "employee", "period", "month", "date", "result_id", "record_id")
for col in work.columns:
    lower = col.lower()
    if any(token in lower for token in numeric_tokens) and not any(token in lower for token in identifier_tokens):
        work[col] = pd.to_numeric(work[col], errors="coerce")

# Reassert that payroll identifiers are preserved before reason/risk logic.
work["payroll_item_id"] = work["payroll_item_id"].astype("string").str.strip()
assert work["payroll_item_id"].notna().all()
assert (work["payroll_item_id"].str.strip().astype(str) != "").all()

def find_col(frame, candidates):
    return first_existing(frame.columns, candidates)

gross_col = find_col(work, ["gross_pay", "gross_salary", "gross", "gross_amount", "total_gross"])
fixed_col = find_col(work, [
    "monthly_fixed_gross",
    "fixed_gross",
    "expected_gross",
    "fixed_salary",
    "expected_salary",
    "base_salary",
])
net_col = find_col(work, ["net_pay", "net_salary", "net", "net_amount", "take_home_pay"])
deduction_col = find_col(work, ["total_deductions", "deductions", "deduction_amount", "total_deduction"])
paid_col = find_col(work, ["paid_amount", "actual_paid", "amount_paid", "payment_amount", "net_paid"])
expected_col = find_col(work, ["expected_pay", "expected_net_pay", "expected_amount", "expected_payment"])
overtime_col = find_col(work, ["overtime_hours", "overtime_pay", "ot_hours", "overtime_amount"])

print("Detected amount fields:", {
    "gross": gross_col, "fixed/expected gross": fixed_col, "net": net_col,
    "deductions": deduction_col, "paid": paid_col, "expected": expected_col,
    "overtime": overtime_col
})

# Use Task 6 rule columns where present, and feature-derived checks as a fallback.
def truthy_col(row, names):
    for name in names:
        if name in row.index and pd.notna(row[name]):
            val = row[name]
            if isinstance(val, str):
                if val.strip().lower() in {"1", "true", "yes", "y"}:
                    return True
            else:
                try:
                    if float(val) == 1:
                        return True
                except (TypeError, ValueError):
                    pass
    return False

def explain_row(row):
    # Rule/model flag names are based on Task 6; missing rule fields simply fall through.
    if truthy_col(row, ["duplicate_payment_rule", "duplicate_payment_flag", "double_payment_flag"]):
        return ("DOUBLE_PAYMENT", "This payroll record matches another payment for the same employee and period.")
    if truthy_col(row, ["negative_net_pay", "negative_net_pay_flag"]) or (net_col and pd.notna(row.get(net_col)) and row[net_col] < 0):
        return ("NEGATIVE_NET_PAY", "The calculated take-home pay is below zero and needs payroll review.")
    if truthy_col(row, ["deductions_over_gross", "deductions_over_gross_flag"]) or (
        gross_col and deduction_col and pd.notna(row.get(gross_col)) and pd.notna(row.get(deduction_col))
        and row[gross_col] >= 0 and row[deduction_col] > row[gross_col]
    ):
        return ("DEDUCTIONS_EXCEED_GROSS", "Recorded deductions are greater than gross pay.")
    if truthy_col(row, ["reconciliation_rule", "net_reconciliation_rule"]) or (
        gross_col and net_col and deduction_col and
        all(pd.notna(row.get(c)) for c in [gross_col, net_col, deduction_col]) and
        abs(float(row[gross_col]) - float(row[deduction_col]) - float(row[net_col])) > 1.0
    ):
        return ("PAYROLL_RECONCILIATION", "Gross pay minus deductions does not match the recorded net pay within ₹1.")
    if gross_col and fixed_col and pd.notna(row.get(gross_col)) and pd.notna(row.get(fixed_col)) and row[fixed_col] > 0:
        ratio = row[gross_col] / row[fixed_col]
        if ratio > 1.5:
            return ("GROSS_ABOVE_EXPECTED", "Gross pay is more than 50% above the expected fixed gross amount.")
        if ratio < 0.5:
            return ("GROSS_BELOW_EXPECTED", "Gross pay is less than half of the expected fixed gross amount.")
    if truthy_col(row, ["zero_pay_rule", "zero_pay_flag"]) or (
        paid_col and pd.notna(row.get(paid_col)) and row[paid_col] == 0
    ):
        return ("ZERO_PAY", "No payment amount is recorded for this payroll period.")
    if truthy_col(row, ["overtime_above_cap", "overtime_rule"]) or (
        overtime_col and pd.notna(row.get(overtime_col)) and row[overtime_col] > 80
    ):
        return ("OVERTIME_ABOVE_CAP", "Recorded overtime is unusually high and should be checked against approved hours.")
    return ("STATISTICAL_OUTLIER", "This payroll record differs from the usual pattern and should be checked against the source records.")

explanations = work.apply(explain_row, axis=1, result_type="expand")
explanations.columns = ["reason_code", "reason_text"]
work[["reason_code", "reason_text"]] = explanations
display(work[["reason_code", "reason_text"]].value_counts().rename("rows").reset_index().head(20))


Detected amount fields: {'gross': 'gross_pay', 'fixed/expected gross': 'monthly_fixed_gross', 'net': 'net_pay', 'deductions': 'total_deductions', 'paid': None, 'expected': None, 'overtime': 'overtime_pay'}


,reason_code,reason_text,rows
0,STATISTICAL_OUTLIER,This payroll record differs from the usual pat...,6520
1,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,495
2,OVERTIME_ABOVE_CAP,Recorded overtime is unusually high and should...,146
3,GROSS_BELOW_EXPECTED,Gross pay is less than half of the expected fi...,114
4,PAYROLL_RECONCILIATION,Gross pay minus deductions does not match the ...,42
5,DOUBLE_PAYMENT,This payroll record matches another payment fo...,30
6,NEGATIVE_NET_PAY,The calculated take-home pay is below zero and...,8


## 5. Estimate money at risk

For a payroll flag, the estimate should reflect the amount potentially overpaid, underpaid, or unreconciled—not the anomaly score.

Priority:
- If both expected and paid amounts are available, use the absolute gap.
- Otherwise, use the absolute payroll reconciliation gap if gross, deductions, and net are available.
- Otherwise, use the difference between gross and fixed/expected gross.
- For a duplicate-payment signal, use the payment amount if present.
- If the necessary amount data is unavailable, keep `money_at_risk` blank. Do not invent an amount.

These estimates should be reviewed by payroll staff before any action is taken.


In [167]:
def money_risk(row):
    code = row["reason_code"]

    # For a duplicate payment, use the take-home pay as a
    # per-record estimate of the potentially duplicated payment.
    if code == "DOUBLE_PAYMENT":
        for col in [net_col, paid_col, gross_col]:
            if col and pd.notna(row.get(col)):
                amount = abs(float(row[col]))
                if amount > 0:
                    return round(amount, 2)
        return np.nan

    # A reconciliation issue needs an actual arithmetic gap.
    if code == "PAYROLL_RECONCILIATION":
        if (
            gross_col and deduction_col and net_col
            and all(pd.notna(row.get(c))
                    for c in [gross_col, deduction_col, net_col])
        ):
            gap = abs(
                float(row[gross_col])
                - float(row[deduction_col])
                - float(row[net_col])
            )
            return round(gap, 2) if gap > 1.00 else np.nan

        return np.nan

    # Compare actual gross with expected/fixed gross only
    # for flags specifically about that difference.
    if code in ["GROSS_ABOVE_EXPECTED", "GROSS_BELOW_EXPECTED"]:
        if (
            gross_col and fixed_col
            and pd.notna(row.get(gross_col))
            and pd.notna(row.get(fixed_col))
        ):
            gap = abs(
                float(row[gross_col]) - float(row[fixed_col])
            )
            return round(gap, 2) if gap > 0 else np.nan

        return np.nan

    # For a zero-pay record, estimate using an expected amount
    # only where a genuine amount field is available.
    if code == "ZERO_PAY":
        for col in [expected_col, fixed_col]:
            if col and pd.notna(row.get(col)):
                amount = abs(float(row[col]))
                if amount > 0:
                    return round(amount, 2)
        return np.nan

    # A negative net-pay result is an amount needing investigation,
    # not proof that the same amount was actually lost.
    if code == "NEGATIVE_NET_PAY":
        if net_col and pd.notna(row.get(net_col)):
            amount = abs(float(row[net_col]))
            return round(amount, 2) if amount > 0 else np.nan

    # Statistical outliers have no direct monetary gap by themselves.
    # Leave money_at_risk blank rather than treating the full salary as a loss.
    return np.nan

work["money_at_risk_numeric"] = work.apply(money_risk, axis=1)
print("Rows with a numeric risk estimate:", int(work["money_at_risk_numeric"].notna().sum()), "of", len(work))
display(work[["reason_code", "money_at_risk_numeric"]].head(10))


Rows with a numeric risk estimate: 689 of 7355


,reason_code,money_at_risk_numeric
0,STATISTICAL_OUTLIER,NaN
1,GROSS_ABOVE_EXPECTED,56126.52
2,STATISTICAL_OUTLIER,NaN
3,GROSS_ABOVE_EXPECTED,12928.21
4,STATISTICAL_OUTLIER,NaN
5,GROSS_ABOVE_EXPECTED,26197.56
6,STATISTICAL_OUTLIER,NaN
7,GROSS_ABOVE_EXPECTED,38412.36
8,STATISTICAL_OUTLIER,NaN
9,GROSS_ABOVE_EXPECTED,32104.08


## 6. Build the official output table

The final CSV uses exactly the same 12 columns and order as `sample_anomaly_results.csv`. For payroll records, `source_dataset` is set to `payroll_monthly_features`. IDs and metadata are carried from Task 6 where available.


In [168]:
# Section 6: Build and save the official output table
def get_series(frame, candidates, default=np.nan):
    col = first_existing(frame.columns, candidates)
    if col:
        return frame[col]
    return pd.Series(default, index=frame.index)

out = pd.DataFrame(index=work.index)
out["result_id"] = get_series(work, ["result_id"], default="")
if out["result_id"].isna().any() or out["result_id"].astype(str).str.strip().eq("").any():
    out["result_id"] = [f"AN-PAY-{i:05d}" for i in range(1, len(out) + 1)]

out["source_dataset"] = "payroll_monthly_features"
out["record_id"] = work["payroll_item_id"].astype("string")
out["employee_id"] = get_series(work, ["employee_id"], default="")
out["period"] = get_series(work, ["period", "pay_period", "month"], default="")
out["anomaly_score"] = pd.to_numeric(
    get_series(work, ["anomaly_score", "combined_anomaly_score", "model_anomaly_score"]),
    errors="coerce",
)

flag_values = work[flag_col]
if pd.api.types.is_bool_dtype(flag_values):
    flagged_bool = flag_values.fillna(False)
elif pd.api.types.is_numeric_dtype(flag_values):
    flagged_bool = pd.to_numeric(flag_values, errors="coerce").fillna(0).ne(0)
else:
    flagged_bool = flag_values.astype("string").str.strip().str.upper().isin(
        ["1", "Y", "YES", "TRUE", "FLAGGED"]
    )

out["flagged"] = np.where(flagged_bool, "Y", "N")
out["reason_code"] = work["reason_code"].where(flagged_bool, "")
out["reason_text"] = work["reason_text"].where(flagged_bool, "")
out["money_at_risk"] = work["money_at_risk_numeric"].where(flagged_bool, np.nan)
out["model_version"] = get_series(
    work, ["model_version"], default="task6-payroll-detector-task7-explanations"
)
out["scored_at"] = get_series(
    work, ["scored_at"], default=pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")
)

# Match the official sample's exact schema and order.
out = out.reindex(columns=OFFICIAL_COLUMNS)
assert out.columns.tolist() == OFFICIAL_COLUMNS, "Output schema/order does not match the sample."
assert out["flagged"].isin(["Y", "N"]).all()
flagged_mask = out["flagged"].eq("Y")
assert out.loc[flagged_mask, "record_id"].notna().all()
assert out.loc[flagged_mask, "record_id"].astype(str).str.strip().ne("").all()
assert out.loc[flagged_mask, "reason_code"].fillna("").astype(str).str.strip().ne("").all()
assert out.loc[flagged_mask, "reason_text"].fillna("").astype(str).str.strip().ne("").all()

# Rank by available money-at-risk estimates; blank estimates remain last.
risk_sort = pd.to_numeric(
    out["money_at_risk"].astype(str).str.replace(",", "", regex=False),
    errors="coerce",
)
out["_risk_sort"] = risk_sort
out = out.sort_values(
    ["_risk_sort", "anomaly_score"],
    ascending=[False, False],
    na_position="last",
).drop(columns="_risk_sort").reset_index(drop=True)

OUTPUT_PATH = ROOT / "reports" / "task7_explained_anomaly_results.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_PATH, index=False, na_rep="")
print("Saved:", OUTPUT_PATH)
print("Rows:", len(out), "| flagged:", int(out["flagged"].eq("Y").sum()))
print("Official columns match:", out.columns.tolist() == OFFICIAL_COLUMNS)
print("Flagged rows missing record IDs:", int(out.loc[out["flagged"].eq("Y"), "record_id"].isna().sum()))
display(out.head(10))


Saved: /workspace/reports/task7_explained_anomaly_results.csv
Rows: 7355 | flagged: 241
Official columns match: True
Flagged rows missing record IDs: 0


,result_id,source_dataset,record_id,employee_id,period,anomaly_score,flagged,reason_code,reason_text,money_at_risk,model_version,scored_at
0,AN-PAY-01001,payroll_monthly_features,PI-000998,EMP-0423,2025-06-01,0.085254,Y,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,442316.88,task6-payroll-detector-task7-explanations,2026-10-27 17:20
1,AN-PAY-03395,payroll_monthly_features,PI-003389,EMP-0445,2025-12-01,0.077560,Y,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,436749.69,task6-payroll-detector-task7-explanations,2026-10-27 17:20
2,AN-PAY-05839,payroll_monthly_features,PI-005825,EMP-0423,2026-06-01,0.083724,Y,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,433891.16,task6-payroll-detector-task7-explanations,2026-10-27 17:20
3,AN-PAY-03310,payroll_monthly_features,PI-003304,EMP-0304,2025-12-01,0.145167,Y,PAYROLL_RECONCILIATION,Gross pay minus deductions does not match the ...,430106.85,task6-payroll-detector-task7-explanations,2026-10-27 17:20
4,AN-PAY-07113,payroll_monthly_features,PI-007098,EMP-0423,2026-09-01,0.082135,Y,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,428084.95,task6-payroll-detector-task7-explanations,2026-10-27 17:20
5,AN-PAY-04898,payroll_monthly_features,PI-004887,EMP-0198,2026-04-01,0.055775,Y,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,421923.06,task6-payroll-detector-task7-explanations,2026-10-27 17:20
6,AN-PAY-03379,payroll_monthly_features,PI-003373,EMP-0423,2025-12-01,0.082288,Y,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,419251.17,task6-payroll-detector-task7-explanations,2026-10-27 17:20
7,AN-PAY-04601,payroll_monthly_features,PI-004590,EMP-0423,2026-03-01,0.079549,Y,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,412976.58,task6-payroll-detector-task7-explanations,2026-10-27 17:20
8,AN-PAY-07144,payroll_monthly_features,PI-007129,EMP-0479,2026-09-01,0.002990,Y,GROSS_BELOW_EXPECTED,Gross pay is less than half of the expected fi...,405319.47,task6-payroll-detector-task7-explanations,2026-10-27 17:20
9,AN-PAY-00100,payroll_monthly_features,PI-000100,EMP-0198,2025-04-01,0.050032,Y,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,367209.18,task6-payroll-detector-task7-explanations,2026-10-27 17:20


## 7. Test 20 flagged rows and produce review evidence

The test confirms that 20 flagged examples have a reason code and a plain-language sentence. It also checks the official schema. The 20-row review CSV can be shown to a friend/colleague for the required understandability check; record their feedback separately in the project report.


In [169]:
flagged_rows = out[out["flagged"].eq("Y")].copy()
if len(flagged_rows) < 20:
    raise ValueError(f"Need at least 20 flagged rows for the Task 7 review; found {len(flagged_rows)}.")

# The official output is already prioritized by estimated risk; blank estimates sort last.
review20 = flagged_rows.head(20).copy()
assert len(review20) == 20
assert review20["reason_code"].astype(str).str.strip().ne("").all()
assert review20["reason_text"].astype(str).str.strip().ne("").all()
assert review20.columns.tolist() == OFFICIAL_COLUMNS

REVIEW_PATH = ROOT / "reports" / "task7_20_flagged_rows_review.csv"
review20.to_csv(REVIEW_PATH, index=False, na_rep="")
print("20-row review saved:", REVIEW_PATH)
display(review20[["result_id", "employee_id", "period", "reason_code", "reason_text", "money_at_risk"]])


20-row review saved: /workspace/reports/task7_20_flagged_rows_review.csv


,result_id,employee_id,period,reason_code,reason_text,money_at_risk
0,AN-PAY-01001,EMP-0423,2025-06-01,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,442316.88
1,AN-PAY-03395,EMP-0445,2025-12-01,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,436749.69
2,AN-PAY-05839,EMP-0423,2026-06-01,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,433891.16
3,AN-PAY-03310,EMP-0304,2025-12-01,PAYROLL_RECONCILIATION,Gross pay minus deductions does not match the ...,430106.85
4,AN-PAY-07113,EMP-0423,2026-09-01,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,428084.95
5,AN-PAY-04898,EMP-0198,2026-04-01,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,421923.06
6,AN-PAY-03379,EMP-0423,2025-12-01,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,419251.17
7,AN-PAY-04601,EMP-0423,2026-03-01,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,412976.58
8,AN-PAY-07144,EMP-0479,2026-09-01,GROSS_BELOW_EXPECTED,Gross pay is less than half of the expected fi...,405319.47
9,AN-PAY-00100,EMP-0198,2025-04-01,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,367209.18


## 8. Quality checks and prioritization summary

When multiple flags have the same reason, higher money at risk should be reviewed first. When money at risk is missing, review flags with direct rule-based reasons before statistical outliers, then use anomaly score as a tie-breaker. This is a practical review policy, not a claim that the anomaly score represents rupees.


In [170]:
# Check schema, reason completeness, and risk availability.
quality_summary = pd.DataFrame({
    "check": [
        "Official output columns match sample",
        "Every flagged row has reason_code",
        "Every flagged row has reason_text",
        "Flagged rows with numeric money_at_risk",
        "Flagged rows with blank money_at_risk",
    ],
    "passed_or_count": [
        out.columns.tolist() == OFFICIAL_COLUMNS,
        bool(out.loc[out["flagged"].eq("Y"), "reason_code"].notna().all()),
        bool(out.loc[out["flagged"].eq("Y"), "reason_text"].astype(str).str.strip().ne("").all()),
        int(pd.to_numeric(out.loc[out["flagged"].eq("Y"), "money_at_risk"].astype(str).str.replace(",", "", regex=False), errors="coerce").notna().sum()),
        int(pd.to_numeric(out.loc[out["flagged"].eq("Y"), "money_at_risk"].astype(str).str.replace(",", "", regex=False), errors="coerce").isna().sum()),
    ]
})
display(quality_summary)

reason_summary = (
    out[out["flagged"].eq("Y")]
    .groupby(["reason_code", "reason_text"], dropna=False)
    .agg(flag_count=("result_id", "count"))
    .reset_index()
    .sort_values("flag_count", ascending=False)
)
display(reason_summary)

risk_numeric = pd.to_numeric(out["money_at_risk"].astype(str).str.replace(",", "", regex=False), errors="coerce")
print("Sum of available risk estimates (not confirmed loss): ₹{:,.2f}".format(risk_numeric.sum(skipna=True)))
print("Reminder: blank risk values mean the input did not support a defensible amount estimate.")


,check,passed_or_count
0,Official output columns match sample,True
1,Every flagged row has reason_code,True
2,Every flagged row has reason_text,True
3,Flagged rows with numeric money_at_risk,190
4,Flagged rows with blank money_at_risk,51


,reason_code,reason_text,flag_count
1,GROSS_ABOVE_EXPECTED,Gross pay is more than 50% above the expected ...,105
5,STATISTICAL_OUTLIER,This payroll record differs from the usual pat...,51
4,PAYROLL_RECONCILIATION,Gross pay minus deductions does not match the ...,42
0,DOUBLE_PAYMENT,This payroll record matches another payment fo...,30
3,NEGATIVE_NET_PAY,The calculated take-home pay is below zero and...,8
2,GROSS_BELOW_EXPECTED,Gross pay is less than half of the expected fi...,5


Sum of available risk estimates (not confirmed loss): ₹25,555,337.18
Reminder: blank risk values mean the input did not support a defensible amount estimate.


## 9. Human review record

After showing `task7_20_flagged_rows_review.csv` to a friend or colleague, record the result here. Do not claim the human review passed unless it actually happened.

- Reviewer (name/role):
- Date:
- Could they explain the reason without help? (Yes / Partly / No)
- Which reason codes were confusing?
- Changes made after feedback:


In [171]:
# Final validation: schema, identifiers, explanations, risk estimates, and review artifact
flagged_mask = out["flagged"].eq("Y")
record_ids = out["record_id"].astype("string").fillna("").str.strip()
risk_numeric = pd.to_numeric(
    out["money_at_risk"].astype(str).str.replace(",", "", regex=False),
    errors="coerce",
)

print("Official columns match:", out.columns.tolist() == OFFICIAL_COLUMNS)
print("Total output rows:", len(out))
print("Flagged rows:", int(flagged_mask.sum()))
print("Flagged rows missing record IDs:", int(record_ids[flagged_mask].eq("").sum()))
print("Flagged rows missing reason codes:", int(out.loc[flagged_mask, "reason_code"].fillna("").astype(str).str.strip().eq("").sum()))
print("Flagged rows missing reason text:", int(out.loc[flagged_mask, "reason_text"].fillna("").astype(str).str.strip().eq("").sum()))
print("Flagged rows with numeric money-at-risk estimates:", int(risk_numeric[flagged_mask].notna().sum()))
print("Flagged rows with blank money-at-risk estimates:", int(risk_numeric[flagged_mask].isna().sum()))
print("Sum of available estimates (not confirmed loss): ₹{:,.2f}".format(risk_numeric.sum(skipna=True)))

print("\\nReason-code counts:")
display(out.loc[flagged_mask, "reason_code"].value_counts().rename_axis("reason_code").reset_index(name="rows"))

print("\\nGenerated files:")
print("Main output exists:", OUTPUT_PATH.exists(), OUTPUT_PATH)

REVIEW_PATH = ROOT / "reports" / "task7_20_flagged_rows_review.csv"
print("20-row review file exists:", REVIEW_PATH.exists(), REVIEW_PATH)
if REVIEW_PATH.exists():
    review_check = pd.read_csv(REVIEW_PATH)
    print("Review row count:", len(review_check))

assert out.columns.tolist() == OFFICIAL_COLUMNS
assert record_ids[flagged_mask].ne("").all()
assert out.loc[flagged_mask, "reason_code"].fillna("").astype(str).str.strip().ne("").all()
assert out.loc[flagged_mask, "reason_text"].fillna("").astype(str).str.strip().ne("").all()
assert OUTPUT_PATH.exists()


Official columns match: True
Total output rows: 7355
Flagged rows: 241
Flagged rows missing record IDs: 0
Flagged rows missing reason codes: 0
Flagged rows missing reason text: 0
Flagged rows with numeric money-at-risk estimates: 190
Flagged rows with blank money-at-risk estimates: 51
Sum of available estimates (not confirmed loss): ₹25,555,337.18
\nReason-code counts:


,reason_code,rows
0,GROSS_ABOVE_EXPECTED,105
1,STATISTICAL_OUTLIER,51
2,PAYROLL_RECONCILIATION,42
3,DOUBLE_PAYMENT,30
4,NEGATIVE_NET_PAY,8
5,GROSS_BELOW_EXPECTED,5


\nGenerated files:
Main output exists: True /workspace/reports/task7_explained_anomaly_results.csv
20-row review file exists: True /workspace/reports/task7_20_flagged_rows_review.csv
Review row count: 20
